# Breast Cancer Classification — Optimized ML Pipeline
Estrategia: ensemble de múltiples modelos con tuning de hiperparámetros y feature engineering.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier
)
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# XGBoost — si no esta instalado: !pip install xgboost
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('XGBoost no disponible, se usara GradientBoosting de sklearn')

print('Librerias cargadas OK')

Librerias cargadas OK


In [2]:
# 1. CARGAR DATOS
df_ori = pd.read_csv('../data/data_breast_cancer_red.csv')
print('Shape total:', df_ori.shape)
print('Columnas:', df_ori.columns.tolist())
print('\nDiagnosis value counts (NaN = test set):')
print(df_ori['diagnosis'].value_counts(dropna=False))

Shape total: (569, 32)
Columnas: ['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']

Diagnosis value counts (NaN = test set):
diagnosis
NaN    250
B      200
M      119
Name: count, dtype: int64


In [3]:
# 2. SEPARAR TRAIN / TEST
df_train = df_ori[df_ori['diagnosis'].notna()].copy()
df_test  = df_ori[df_ori['diagnosis'].isna()].copy()

# Codificar etiqueta si es string (M/B -> 1/0)
le = LabelEncoder()
if not pd.api.types.is_numeric_dtype(df_train['diagnosis']):
    df_train['diagnosis'] = le.fit_transform(df_train['diagnosis'])
    label_classes = le.classes_
else:
    label_classes = None

# Features: excluir 'id' y 'diagnosis'
drop_cols = ['diagnosis'] + [col for col in ['id'] if col in df_ori.columns]
feature_cols = [c for c in df_ori.columns if c not in drop_cols]

X      = df_train[feature_cols].values
y      = df_train['diagnosis'].values
X_test = df_test[feature_cols].values

print(f'Train: {X.shape[0]} muestras | Test: {X_test.shape[0]} muestras | Features: {X.shape[1]}')

Train: 319 muestras | Test: 250 muestras | Features: 30

In [4]:
# 3. PREPROCESAMIENTO
imputer    = SimpleImputer(strategy='median')
X_imp      = imputer.fit_transform(X)
X_test_imp = imputer.transform(X_test)

scaler     = StandardScaler()
X_scaled   = scaler.fit_transform(X_imp)
X_test_sc  = scaler.transform(X_test_imp)

print('Imputacion y escalado aplicados')

Imputacion y escalado aplicados


In [5]:
# 4. EVALUAR MODELOS CON CROSS-VALIDATION (10-fold estratificado)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

boost_model = (
    XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42
    )
    if HAS_XGB else
    GradientBoostingClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=42
    )
)

models = {
    'RandomForest':       RandomForestClassifier(n_estimators=500, max_features='sqrt',
                                                  random_state=42, n_jobs=-1),
    'GBoost/XGB':         boost_model,
    'SVM_RBF':            SVC(C=10, gamma='scale', kernel='rbf', probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'KNN':                KNeighborsClassifier(n_neighbors=5, weights='distance'),
}

results = {}
needs_scale = {'SVM_RBF', 'LogisticRegression', 'KNN'}

for name, model in models.items():
    X_cv = X_scaled if name in needs_scale else X_imp
    scores = cross_val_score(model, X_cv, y, cv=cv, scoring='accuracy')
    results[name] = scores
    print(f'{name:22s}  CV mean: {scores.mean():.4f}  std: {scores.std():.4f}')

RandomForest            CV mean: 0.9528  std: 0.0405


GBoost/XGB              CV mean: 0.9499  std: 0.0375


SVM_RBF                 CV mean: 0.9497  std: 0.0529


LogisticRegression      CV mean: 0.9749  std: 0.0306


KNN                     CV mean: 0.9593  std: 0.0420


In [6]:
# 5. SOFT VOTING ENSEMBLE
rf  = RandomForestClassifier(n_estimators=500, max_features='sqrt', random_state=42, n_jobs=-1)
gb  = boost_model
svm = Pipeline([('sc', StandardScaler()), ('clf', SVC(C=10, gamma='scale', probability=True, random_state=42))])
lr  = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=42))])
knn = Pipeline([('sc', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=5, weights='distance'))])

voting_clf = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('svm', svm), ('lr', lr), ('knn', knn)],
    voting='soft', n_jobs=1
)

voting_scores = cross_val_score(voting_clf, X_imp, y, cv=cv, scoring='accuracy')
print(f'Soft Voting Ensemble  CV mean: {voting_scores.mean():.4f}  std: {voting_scores.std():.4f}')

Soft Voting Ensemble  CV mean: 0.9718  std: 0.0260


In [7]:
# 6. STACKING (meta-learner = LogisticRegression)
stacking_clf = StackingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('svm', svm), ('knn', knn)],
    final_estimator=LogisticRegression(C=1.0, max_iter=1000),
    cv=5, passthrough=False, n_jobs=1
)

stacking_scores = cross_val_score(stacking_clf, X_imp, y, cv=cv, scoring='accuracy')
print(f'Stacking Ensemble     CV mean: {stacking_scores.mean():.4f}  std: {stacking_scores.std():.4f}')

Stacking Ensemble     CV mean: 0.9561  std: 0.0400


In [8]:
# 7. ELEGIR EL MEJOR MODELO Y ENTRENAR CON TODO EL TRAIN
all_results = {
    **{k: v.mean() for k, v in results.items()},
    'Voting Ensemble':   voting_scores.mean(),
    'Stacking Ensemble': stacking_scores.mean(),
}

best_name  = max(all_results, key=all_results.get)
best_score = all_results[best_name]
print(f'Mejor modelo: {best_name}  (CV accuracy: {best_score:.4f})')

model_map = {
    **models,
    'Voting Ensemble':   voting_clf,
    'Stacking Ensemble': stacking_clf,
}
best_model = model_map[best_name]

# Seleccionar datos correctos segun tipo de modelo
if best_name in needs_scale:
    X_fit, X_test_fit = X_scaled, X_test_sc
else:
    X_fit, X_test_fit = X_imp, X_test_imp  # ensembles llevan scaler interno

best_model.fit(X_fit, y)
print('Modelo entrenado con todo el conjunto de entrenamiento')

Mejor modelo: LogisticRegression  (CV accuracy: 0.9749)
Modelo entrenado con todo el conjunto de entrenamiento


In [9]:
import os
os.makedirs('../outputs', exist_ok=True)
OUT_PATH = '../outputs/predictions_ensemble.csv'
# 8. GENERAR PREDICCIONES Y EXPORTAR CSV
y_pred = best_model.predict(X_test_fit)

if label_classes is not None:
    y_pred_labels = le.inverse_transform(y_pred)
else:
    y_pred_labels = y_pred

id_col = df_test['id'] if 'id' in df_test.columns else df_test.index
df_pred_test = pd.DataFrame({'id': id_col, 'diagnosis': y_pred_labels})
df_pred_test.to_csv(OUT_PATH, index=False)

print(f'Archivo {OUT_PATH} generado con {len(df_pred_test)} predicciones')
print(df_pred_test['diagnosis'].value_counts())
df_pred_test.head(10)

Archivo ../outputs/predictions_ensemble.csv generado con 250 predicciones
diagnosis
B    158
M     92
Name: count, dtype: int64


,id,diagnosis
2,84300903,M
4,84358402,M
8,844981,M
9,84501001,M
10,845636,M
12,846226,M
14,84667401,M
15,84799002,M
16,848406,M
17,84862001,M


## Resumen de estrategia

| Tecnica | Por que ayuda |
|---|---|
| Imputacion mediana | Rellena NaNs en features sin sesgar la distribucion |
| StandardScaler | Necesario para SVM, LR y KNN |
| RandomForest (500 arboles) | Robusto, maneja bien features correladas |
| XGBoost/GradientBoosting | Generalmente el mejor modelo unico para tabular |
| SVM RBF | Excelente en datasets pequeños/medianos |
| Soft Voting | Promedia probabilidades, reduce varianza |
| Stacking | Meta-learner aprende como combinar predicciones |
| CV estratificada 10-fold | Estimacion honesta del performance real |